# RawGrad — Backpropagation

This notebook demonstrates backpropagation using the RawGrad neural-network framework.

We will:

- Build a small neural network
- Perform a forward pass
- Calculate classification loss
- Perform the backward pass
- Inspect gradients
- Verify that gradients are produced for trainable layers

The goal is to understand how the neural network calculates gradients that can later be used to update its parameters.

In [1]:
import sys
from pathlib import Path

import numpy as np

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))

from rawgrad.layers import Dense
from rawgrad.activations import ReLU, Softmax
from rawgrad.model import Sequential
from rawgrad.classification_losses import CategoricalCrossEntropy

In [2]:
import sys
from pathlib import Path

import numpy as np

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))

from rawgrad.layers import Dense
from rawgrad.activations import ReLU, Softmax
from rawgrad.model import Sequential
from rawgrad.classification_losses import CategoricalCrossEntropy

In [3]:
np.random.seed(42)

X = np.array([
    [1.0, 2.0, 3.0],
    [2.0, 1.0, 0.5],
    [0.0, 1.0, 2.0],
    [3.0, 2.0, 1.0],
])

labels = np.array([0, 1, 2, 1])

targets = np.zeros((len(labels), 3))
targets[np.arange(len(labels)), labels] = 1.0

print("Input shape:", X.shape)
print("Target shape:", targets.shape)
print("Target labels:", labels)

Input shape: (4, 3)
Target shape: (4, 3)
Target labels: [0 1 2 1]


In [4]:
model = Sequential([
    Dense(3, 8, initialization="xavier"),
    ReLU(),
    Dense(8, 3, initialization="xavier"),
    Softmax(),
])

predictions = model.forward(X)

print("Prediction shape:", predictions.shape)
print("Predictions:")
print(predictions)

Prediction shape: (4, 3)
Predictions:
[[0.75105826 0.08596561 0.16297612]
 [0.3066841  0.50534043 0.18797547]
 [0.62898713 0.1462154  0.22479747]
 [0.39594696 0.43621342 0.16783962]]


In [5]:
loss_function = CategoricalCrossEntropy()

loss = loss_function.forward(predictions, targets)

print("Loss:", loss)

Loss: 0.8227435211326628


In [6]:
loss_gradient = loss_function.backward()

print("Loss gradient shape:", loss_gradient.shape)
print("Loss gradient:")
print(loss_gradient)

Loss gradient shape: (4, 3)
Loss gradient:
[[-0.33286366 -0.         -0.        ]
 [-0.         -0.49471601 -0.        ]
 [-0.         -0.         -1.11211216]
 [-0.         -0.57311396 -0.        ]]


In [7]:
model.backward(loss_gradient)

print("Backward pass completed.")

Backward pass completed.


In [8]:
for index, layer in enumerate(model.layers):
    if isinstance(layer, Dense):
        print(f"Dense layer {index}:")
        print("  Weight gradient shape:", layer.weights_gradient.shape)
        print("  Bias gradient shape:", layer.biases_gradient.shape)
        print("  Weight gradient finite:", np.all(np.isfinite(layer.weights_gradient)))
        print("  Bias gradient finite:", np.all(np.isfinite(layer.biases_gradient)))

Dense layer 0:
  Weight gradient shape: (3, 8)
  Bias gradient shape: (8,)
  Weight gradient finite: True
  Bias gradient finite: True
Dense layer 2:
  Weight gradient shape: (8, 3)
  Bias gradient shape: (3,)
  Weight gradient finite: True
  Bias gradient finite: True


In [9]:
dense_layers = [
    layer for layer in model.layers
    if isinstance(layer, Dense)
]

assert len(dense_layers) == 2

for layer in dense_layers:
    assert layer.weights_gradient.shape == layer.weights.shape
    assert layer.biases_gradient.shape == layer.biases.shape
    assert np.all(np.isfinite(layer.weights_gradient))
    assert np.all(np.isfinite(layer.biases_gradient))

print("BACKPROPAGATION: PASS")

BACKPROPAGATION: PASS
